# ScamGuard — Exploratory Data Analysis (EDA)

**Notebook 01:** ทำความเข้าใจ Dataset ก่อนเริ่ม Train Model

---
**สิ่งที่จะ explore:**
1. Dataset overview & class distribution
2. Text length analysis
3. Signal feature analysis (URL count, phone count, etc.)
4. Word frequency & TF-IDF insights
5. Sample messages by class


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

# Thai font support
matplotlib.rcParams['font.family'] = 'Tahoma'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded ✓')

## 1. Generate and Load Dataset

In [ ]:
from src.data.generator import generate_dataset
from src.data.preprocessor import ThaiTextPreprocessor, preprocess_dataframe

# Generate dataset
df = generate_dataset(n_ham=500, n_spam=400, n_phishing=300, random_state=42)
print(f'Total records: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
df.head(5)

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = {'ham': '#4caf50', 'spam': '#ff9800', 'phishing': '#f44336'}
counts = df['label_name'].value_counts()
ax1 = axes[0]
bars = ax1.bar(counts.index, counts.values, color=[colors[l] for l in counts.index])
ax1.set_title('Class Distribution (Count)', fontsize=14)
ax1.set_xlabel('Label')
ax1.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', fontsize=12, fontweight='bold')

# Pie chart
ax2 = axes[1]
ax2.pie(counts.values, labels=counts.index,
        colors=[colors[l] for l in counts.index],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
ax2.set_title('Class Distribution (%)', fontsize=14)

plt.tight_layout()
plt.savefig('../data/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(counts)

## 3. Text Length Analysis

In [ ]:
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

print('Text Length Statistics by Class:')
print(df.groupby('label_name')[['text_length', 'word_count']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in colors.items():
    subset = df[df['label_name'] == label]
    axes[0].hist(subset['text_length'], alpha=0.6, label=label, color=color, bins=30)
    axes[1].hist(subset['word_count'], alpha=0.6, label=label, color=color, bins=20)

axes[0].set_title('Text Length Distribution')
axes[0].set_xlabel('Number of Characters')
axes[0].legend()

axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Number of Words')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Signal Features Analysis

In [ ]:
preprocessor = ThaiTextPreprocessor(remove_stopwords=False)
signals_df = preprocessor.extract_signals_batch(df['text'].tolist())
df_with_signals = pd.concat([df, signals_df], axis=1)

signal_cols = ['url_count', 'phone_count', 'exclamation_count', 'number_ratio']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), signal_cols):
    grouped = df_with_signals.groupby('label_name')[col].mean()
    bars = ax.bar(grouped.index, grouped.values,
                  color=[colors[l] for l in grouped.index])
    ax.set_title(f'Average {col} by Class')
    ax.set_xlabel('Class')
    for bar, val in zip(bars, grouped.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/eda_signal_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('Signal Feature Averages by Class:')
print(df_with_signals.groupby('label_name')[signal_cols].mean().round(4))

## 5. Sample Messages

In [ ]:
print('=== ตัวอย่าง Ham Messages (ปกติ) ===')
for text in df[df['label_name']=='ham']['text'].sample(3, random_state=42):
    print(f'  ✅ {text[:100]}')

print('\n=== ตัวอย่าง Spam Messages ===')
for text in df[df['label_name']=='spam']['text'].sample(3, random_state=42):
    print(f'  ⚠️ {text[:100]}')

print('\n=== ตัวอย่าง Phishing Messages ===')
for text in df[df['label_name']=='phishing']['text'].sample(3, random_state=42):
    print(f'  🚨 {text[:100]}')

## 6. Key Insights

**สรุปผลจาก EDA:**

1. **Class Distribution:** Dataset มีความสมดุลพอควร
2. **Text Length:** Spam/Phishing มักยาวกว่า Ham เล็กน้อย
3. **URL Count:** Spam/Phishing มี URL มากกว่า Ham อย่างชัดเจน
4. **Phone Count:** Phishing มักมีหมายเลขโทรศัพท์มากที่สุด
5. **Exclamation Marks:** Spam ใช้ ! มากกว่า Ham และ Phishing

**→ Signal features จะช่วย model ได้มากในการ classify**
